In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
%run "./00_setup.ipynb"

In [ ]:
print("## Hold-out 最終評価")
print("テスト期間: 2022-01-01 〜 2024-12-31 (3年間)")
print("")
print("このノートブックは1回のみ実行すること。")
print("複数回実行すると Hold-out データへのリークが発生する。")
print("")

try:
    engine = get_engine()
    print("DB接続成功 — データをロード中...")
    # Note: 実際のロードは BacktestEngine.run() が行う
except Exception as e:
    print(f"DB接続失敗: {e}")
    print("実行には PostgreSQL 接続が必要です。")

In [ ]:
print("""
## バックテスト実行

from backtest.engine import BacktestEngine
from backtest.validation_suite import BacktestValidationSuite
from pipelines.training_pipeline import TrainingPipelineV5

# 1. 学習済みモデルをロード (MLflow または直接)
# models = load_trained_models()  # MLflow からロード
# または TrainingPipelineV5 で新規学習:
#   models = TrainingPipelineV5().run("2015-01-01", "2021-12-31")

# 2. バックテスト実行
# engine = BacktestEngine(models, initial_bankroll=100000)
# result = engine.run("2022-01-01", "2024-12-31")

# 3. 結果の確認
# print(f"Total bets: {result.total_bets}")
# print(f"Total ROI: {result.total_roi:.3%}")
# print(f"Max drawdown: {result.max_drawdown:.3%}")
# print(f"Final bankroll: {result.final_bankroll:,.0f}円")
""")

In [ ]:
print("""
## §13.2 合格基準 (v5.0)

| 基準 | 閾値 | 必須 |
|------|------|------|
| 複勝回収率 | >= 100% | YES |
| ワイド回収率 | >= 103% | YES |
| 全体回収率 | >= 101% | YES |
| 最大ドローダウン | <= 16% | YES |
| 月次100%超 | >= 22/36ヶ月 | YES |

※ 全て満たすことが本運用移行の前提条件。
""")

In [ ]:
print("""
## §13.2 追加合格条件 (v5.1)

| 基準 | 閾値 | 必須 |
|------|------|------|
| EV補正モデルのMAE改善 | >= 10% | YES |
| 中穴ゾーンEV誤差改善 | >= 15% | YES |
| log_error SHAP寄与度 | > 0 | YES |

EV補正の有効性を定量的に確認する。
""")

In [ ]:
print("""
## §13.2 追加合格条件 (v5.4)

| 基準 | 閾値 | 必須 |
|------|------|------|
| P補正AUC改善 | >= 1% | YES |
| P/E補正相関 | < 0.3 | YES |
| E補正MAE改善 (winner) | 改善 | YES |
| Wide Var_proxy 正確性 | EV/(E×sqrt(P)) 一致 | YES |

P/E分解補正の有効性を確認する。
""")

In [ ]:
print("""
## 月次ROI推移

36ヶ月の月次ROIを棒グラフで表示:
  - 緑色: ROI >= 100% (黒字月)
  - 赤色: ROI < 100% (赤字月)
  - 22ヶ月以上が緑色であること

実装:
  result.monthly_returns を DataFrame に変換
  plt.bar(months, roi_values, color=conditions)
  plt.axhline(y=1.0, color='black', linestyle='--', alpha=0.5)
""")

In [ ]:
print("""
## Bankroll 遷移

全期間の bankroll 推移を折れ線グラフで表示:
  - 初期資金: 100,000円
  - max drawdown 領域を赤で塗りつぶし
  - RecoveryState (NORMAL/REDUCED/RECOVERING) を背景色で表示

実装:
  bet_history から cumsum で bankroll 遷移を計算
  plt.fill_between で DD 領域を描画
""")

In [ ]:
print("""
## 最終判定

全合格基準を満たした場合:
  → 「本運用への移行可」が出力される
  → 500円/bet での小額実運用を開始

合格基準を満たさなかった場合:
  → 不合格の項目を特定
  → モデル/特徴量/パラメータの調整が必要
  → TrainingPipelineV5 で再学習

⚠️ 重要: Hold-out 期間は 1回のみ使用すること。
   複数回の実行は Hold-out の意味を失う。
""")